# Libraries

In [2]:
import os
import shutil

import pandas as pd
import geopandas as gpd
import rasterio as rio
import pickle
import osmnx as ox
import networkx as nx
import numpy as np
import math

from pyproj import Transformer
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Set home directory
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data' # CURA
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data' # SCARP
# home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data' # Home

In [60]:
'''Housekeeping items'''
# Make a copy of networks that are affected by hurricane CAT1-5
coastal_networkID_df = pd.read_csv(home_dir + '/Coastal_flooding/network_polys_IDs.csv')

# Array of network ids
netids = coastal_networkID_df.net_id.values.astype(int)

# Copy graphs
source_folder = home_dir + '/all_graph_flooded/'
target_folder = home_dir + '/Coastal_flooding/graphs/'

# Find networks in netids that are affected by coastal flooding and make a copy to new location
for file in os.listdir(source_folder):
    if file.endswith('.pk'):
        netid = int(file.split('_')[4])
        if netid in netids:
            # make a copy
            shutil.copy2(os.path.join(home_dir,source_folder, file),
                         os.path.join(home_dir,target_folder, file))
        else:
            continue

# Attach inundation values

In [134]:
# Location of graphs
graphs_dir = home_dir + '/Coastal_flooding/graphs/'

# Location of coastal flood risk maps
coastal_flooding_dir = '/Users/yiyi/Desktop/Coastal Flooding/NOAA_National_storm_surge/US_SLOSH_MOM_Inundation_20250923'

hurricane_categories = [1,2,3,4,5]

hurricane_rasters = {}
hurricane_bands = {}

for hurricane_category in hurricane_categories:
    # coastal flood risk map raster
    src = rio.open(f'/Users/yiyi/Desktop/Coastal Flooding/NOAA_National_storm_surge/US_SLOSH_MOM_Inundation_20250923/us_Category{hurricane_category}_MOM_Inundation_HIGH.tif')

    hurricane_rasters[hurricane_category] = src
    hurricane_bands[hurricane_category] = src.read(1) # load raster band
# Transform to lat lon
transformer = Transformer.from_crs(
    "EPSG:4326",      # lon/lat
    src.crs,  # EPSG:4269
    always_xy=True
)

**Dictionary for NOAA Hurricane risk maps**  
Value: description  
1: 00 to 01 foot above ground<br>
2: 01 to 02 feet above ground<br>
3: 02 to 03 feet above ground<br>
...<br>
20: 19 to 20 feet above ground<br>
21: Greater than 20 feet above ground<br>
99: Levee Areas - Consult Local Officials for flood risk<br>

In [142]:
for file in tqdm(os.listdir(graphs_dir)):
    if file.endswith('.pk'):
        # Read in the graph pickle file
        g = pickle.load(open(graphs_dir + file, 'rb'))
        # Network id
        net_id = int(file.split("_")[4])

        #
        for node, data in g.nodes(data=True):
            lon = data["x"]
            lat = data["y"]

            # Reproject lon/lat to raster crs
            x, y = transformer.transform(lon, lat)

            # Iterate through hurricane categories
            for hurricane_category in hurricane_categories:
                # Load hurricane raster, band
                src = hurricane_rasters[hurricane_category]
                band = hurricane_bands[hurricane_category]
                # Get raster value at node
                try:
                    row, col = src.index(x, y)
                    value = band[row, col]

                    if nodata is not None and value == nodata:
                        value = np.nan

                except IndexError:
                    value = np.nan  # node outside raster extent
                # Attach raster value to node attribute
                g.nodes[node][f"CAT_{hurricane_category}_value"] = value

        # Save the graph
        with open(home_dir + f"/Coastal_flooding/graphs_node_flooded/net_{net_id}.pk", "wb") as f:
            pickle.dump(g, f, protocol=2)

100%|██████████| 56/56 [03:11<00:00,  3.43s/it]


In [182]:
# Add inundation values on graph edges
def take_val(i, j):
    if math.isnan(i) and math.isnan(j):
        return float("nan")
    if math.isnan(i):
        return j
    if math.isnan(j):
        return i
    return max(i, j)

graph_flood_dir = home_dir + '/Coastal_flooding/graphs_node_flooded/'
for file in tqdm(os.listdir(graph_flood_dir)):
    if file.endswith('.pk'):
        net_id = int("".join(filter(str.isdigit, file)))
        # Read in the graph pickle file
        g = pickle.load(open(graph_flood_dir + file, 'rb'))
        for i,j,data in g.edges.data():
            for hurricane_category in hurricane_categories:
                i_val = g.nodes[i][f'CAT_{hurricane_category}_value']
                j_val = g.nodes[j][f'CAT_{hurricane_category}_value']
                val = take_val(i_val,j_val)
                g.edges[i, j, 0][f'CAT_{hurricane_category}_value'] = val
        # Save the graph
        with open(home_dir + f"/Coastal_flooding/graphs_node_edge_flooded/net_{net_id}.pk", "wb") as f:
            pickle.dump(g, f, protocol=2)

# Direct exposure

In [244]:
hurricane_categories = [1,2,3,4,5]

net_id_col = []
total_length_col = []
cat1_expo_col = []
cat2_expo_col = []
cat3_expo_col = []
cat4_expo_col = []
cat5_expo_col = []
cat_lsts = [cat1_expo_col, cat2_expo_col, cat3_expo_col, cat4_expo_col, cat5_expo_col]
# 
for file in tqdm(os.listdir(home_dir + "/Coastal_flooding/graphs_node_edge_flooded/")):

    net_id = int("".join(filter(str.isdigit, file)))
    net_id_col.append(net_id)
    g = pickle.load(open(home_dir + "/Coastal_flooding/graphs_node_edge_flooded/"+ file, 'rb'))
    
    total_length = 0
    for i,j,data in g.edges.data():
        total_length += data["length"]
    total_length_col.append(total_length)

    for hurricane_category in hurricane_categories:
        
        exposure_length = 0
        
        for i,j,data in g.edges.data():

            try:

                if math.isnan(data[f"CAT_{hurricane_category}_value"]):
                    continue
                elif data[f"CAT_{hurricane_category}_value"]>1:
                    exposure_length += data["length"]
            except KeyError:
                continue

        cat_lsts[hurricane_category-1].append(exposure_length)
    
exposure_data = {
    "net_id": net_id_col,
    "total_length" : total_length_col,
    "CAT1_expo" : cat1_expo_col,
    "CAT2_expo" : cat2_expo_col,
    "CAT3_expo" : cat3_expo_col,
    "CAT4_expo" : cat4_expo_col,
    "CAT5_expo" : cat5_expo_col
}

exposure_data_df = pd.DataFrame.from_dict(exposure_data)

exposure_data_df.head(3)

# Calculate exposure percentage
for hurricane_category in hurricane_categories:
    exposure_data_df[f'CAT{hurricane_category}_expo_perc'] = 100*exposure_data_df[f'CAT{hurricane_category}_expo']/exposure_data_df['total_length']
    
exposure_data_df.to_csv(home_dir + "/Coastal_flooding/graph_exposure.csv")

100%|██████████| 55/55 [01:00<00:00,  1.09s/it]


# Indirect impact

Original OD pairs

In [20]:
# 
exposure_df = pd.read_csv(home_dir + '/Coastal_flooding/graph_exposure.csv', index_col=0)
net_ids = list(exposure_df.net_id)

for file in os.listdir(home_dir + '/ODpts_propbability/'):
    if file.endswith('.txt'):
        net_id = int(file.split('_')[-1].split(".")[0])
        if net_id in net_ids:
            # Copy the file to new location
            shutil.copy(home_dir + '/ODpts_propbability/' + file,
                        home_dir + '/Coastal_flooding/OD_pt_probability/')